# Brent: итоговый исследовательский ноутбук

Этот ноутбук собирает уже готовые результаты этапов 1–4 без нового обучения.
Он читает сохранённые артефакты и использует фактические реализации `stage2` и `stage3`,
не пересматривая методологию и не меняя рабочий код.

In [1]:
import sys
from pathlib import Path
import json
import warnings

import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
if not (ROOT / 'task1_brent').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert (ROOT / 'task1_brent').is_dir(), 'Запустите ноутбук из корня репозитория'
B = ROOT / 'task1_brent'

from task1_brent.brent_lib.data import load_brent
from task1_brent.brent_lib.stage2 import (
    build_stage2_inventory_summary,
    calculate_stage2_track_leaderboard,
)
from task1_brent.brent_lib.stage3 import (
    build_stage3_common_origin_summary,
    build_stage3_probabilistic_leaderboard,
)

RU_COLUMNS = {
    'first_date': 'Первая дата',
    'last_date': 'Последняя дата',
    'n_months': 'Число месяцев',
    'stage4_mode': 'Режим этапа 4',
    'stage4_stride_months': 'Шаг точек прогноза, мес.',
    'stage4_epochs_max': 'Максимум эпох',
    'selection_rule': 'Правило выбора',
    'model': 'Модель',
    'horizon': 'Горизонт',
    'n': 'Число прогнозов',
    'rel_MAE_vs_naive': 'Отн. MAE к naive',
    'relative_MAE': 'Отн. MAE',
    'MAPE_%': 'MAPE, %',
    'name': 'Показатель',
    'source_file': 'Файл-источник',
    'frequency': 'Частота',
    'aggregation': 'Агрегация',
    'release_lag_months': 'Лаг публикации, мес.',
    'classification': 'Класс',
    'track': 'Набор признаков',
    'feature_set': 'Набор признаков',
    'n_forecasts': 'Число прогнозов',
    'internal_n': 'Внутренние признаки: n',
    'external_core_n': 'Внешние признаки: n',
    'common_n': 'Общее n',
    'first_origin': 'Первая точка прогноза',
    'last_origin': 'Последняя точка прогноза',
    'same_origins_internal_external': 'Совпадают точки прогноза',
    'coverage_90': 'Покрытие 90%',
    'width_90': 'Ширина 90%',
    'median_MAE': 'MAE медианы',
    'median_RMSE': 'RMSE медианы',
    'crossing_rate_before': 'Доля пересечений до коррекции',
    'stride_months': 'Шаг, мес.',
    'n_test_origins': 'Число тестовых точек',
    'n_forecasts_per_horizon': 'Прогнозов на горизонт',
    'note': 'Примечание',
    'n_origins': 'Число точек прогноза',
    'mode': 'Режим',
    'epochs_max': 'Максимум эпох',
    'models_in_test': 'Моделей в тесте',
    'family': 'Семейство',
    'best_model': 'Лучшая модель',
    'evaluation_sample': 'Выборка оценки',
    'metric': 'Метрика',
    'value': 'Значение',
    'comparability': 'Сопоставимость',
    'available_classical_models': 'Доступные классические модели',
    'available_direct_models': 'Доступные прямые модели',
}

RU_VALUES = {
    'internal': 'внутренние',
    'external_core': 'основные внешние',
    'sensitivity': 'чувствительность',
    'reasonable': 'основной',
    'daily': 'дневная',
    'monthly': 'месячная',
    'monthly_mean': 'среднее за месяц',
    'monthly_std_log_return': 'месячная волатильность лог-доходности',
    'as_is': 'без агрегации',
    'NOT DIRECTLY COMPARABLE': 'НАПРЯМУЮ НЕСОПОСТАВИМО',
    'Stage 1': 'Этап 1',
    'Stage 2': 'Этап 2',
    'Stage 3': 'Этап 3',
    'Stage 4': 'Этап 4',
}

def display_ru(frame):
    out = frame.rename(columns=RU_COLUMNS).copy()
    for col in out.select_dtypes(include='object').columns:
        out[col] = out[col].replace(RU_VALUES)
    for col in out.select_dtypes(include='bool').columns:
        out[col] = out[col].map({True: 'Да', False: 'Нет'})
    display(out)

series = load_brent()
stage4_meta = json.loads((B / 'outputs' / 'stage4_eval' / 'metadata.json').read_text())
selection_rule_ru = {
    'Validation MAE separately per horizon; frozen for test.':
        'Минимум MAE на валидации отдельно по каждому горизонту; выбор фиксируется до теста.'
}.get(stage4_meta['selection_rule'], stage4_meta['selection_rule'])

summary = pd.DataFrame([{
    'Первая дата': str(series.index[0].date()),
    'Последняя дата': str(series.index[-1].date()),
    'Число месяцев': len(series),
    'Режим этапа 4': stage4_meta['mode'],
    'Шаг точек прогноза, мес.': stage4_meta['stride_months'],
    'Максимум эпох': stage4_meta['epochs_max'],
    'Правило выбора': selection_rule_ru,
}])
display(summary)


,Первая дата,Последняя дата,Число месяцев,Режим этапа 4,"Шаг точек прогноза, мес.",Максимум эпох,Правило выбора
0,1987-05-01,2026-03-01,467,быстрый,12,20,Минимум MAE на валидации по горизонту; выбор зафиксирован ...


## 1) Область исследования, данные и протокол оценки

- Источник: месячные цены Brent по историческому срезу до 2026-03.
- Признаки: лог-цена и признаки из этапов 1–4 без случайного перемешивания и утечки информации из будущего.
- Бэктест и выбор моделей: календарное разбиение; валидация используется для выбора, тест — только для окончательной ретроспективной оценки.
- Новые эксперименты здесь не запускаются: ноутбук собирает уже полученные артефакты и результаты.

In [2]:
validation = pd.read_csv(B / 'outputs' / 'validation_metrics.csv')
key_h = [1, 3, 6, 12]
validation_table = validation[validation['horizon'].isin(key_h)].sort_values(['horizon', 'MAE']).copy()
display_ru(validation_table[['model', 'horizon', 'n', 'MAE', 'RMSE', 'rel_MAE_vs_naive']].round(3))

selection = pd.read_csv(B / 'outputs' / 'selection_development.csv')
display(Markdown('**Итоговый выбор по выборке разработки:**'))
display_ru(selection.head(10))


,Модель,Горизонт,n,MAE,RMSE,Отн. MAE к naive
0,mean_reversion,1,50,5.490,7.820,0.971
1,naive,1,50,5.653,7.888,1.000
2,ets_log,1,50,5.656,7.889,1.000
3,drift,1,50,5.664,7.892,1.002
4,hgb_direct,1,50,6.103,8.339,1.080
5,ensemble,1,50,6.388,9.105,1.130
6,ridge_direct,1,50,12.033,16.348,2.129
14,mean_reversion,3,48,7.213,10.508,0.873
15,naive,3,48,8.265,11.324,1.000
16,ets_log,3,48,8.306,11.356,1.005


**Итоговый выбор по выборке разработки:**

,Горизонт,Модель
0,1,mean_reversion
1,2,mean_reversion
2,3,mean_reversion
3,4,mean_reversion
4,5,mean_reversion
5,6,mean_reversion
6,7,mean_reversion
7,8,mean_reversion
8,9,mean_reversion
9,10,mean_reversion


## 1A) Данные и разведочный анализ

- Первая дата: самый ранний месяц в загруженном ряду Brent.
- Последняя дата: последнее наблюдение в исторической выборке.
- Размер выборки: число месячных наблюдений с проверкой пропусков и дубликатов.
- Разведочный анализ ограничен тремя графиками: уровень цены, лог-доходность и скользящая волатильность.

In [3]:
inventory = build_stage2_inventory_summary()
display_ru(inventory.head(10).reset_index(drop=True))

internal = calculate_stage2_track_leaderboard(series, feature_set='internal', horizons=(1, 3, 6, 12))
external = calculate_stage2_track_leaderboard(series, feature_set='external_core', horizons=(1, 3, 6, 12))
sensitivity = calculate_stage2_track_leaderboard(series, feature_set='sensitivity', horizons=(1, 3, 6, 12))
stage2_summary = pd.concat([
    internal.assign(track='INTERNAL'),
    external.assign(track='EXTERNAL_CORE'),
    sensitivity.assign(track='SENSITIVITY'),
], ignore_index=True)
stage2_summary['track'] = stage2_summary['track'].replace({
    'INTERNAL': 'Внутренние признаки',
    'EXTERNAL_CORE': 'Основные внешние',
    'SENSITIVITY': 'Чувствительность',
})
display(Markdown('**Внутренние признаки vs основные внешние vs чувствительность:**'))
display_ru(
    stage2_summary.sort_values(['track', 'horizon', 'MAE'])[
        ['track', 'model', 'horizon', 'MAE', 'RMSE', 'rel_MAE_vs_naive', 'n_forecasts']
    ].round(3).head(24)
)


,Показатель,Файл-источник,Частота,Агрегация,"Лаг публикации, мес.",Класс
0,usd_rub,usd_rub.xlsx,дневная,среднее за месяц,1,основной
1,fx_volatility,usd_rub.xlsx,дневная,месячная волатильность лог-доходности,1,основной
2,us_cpi,us_cpi.xlsx,месячная,без агрегации,1,основной
3,euro_area_cpi,euro_area_cpi.xlsx,месячная,без агрегации,1,основной
4,ruonia,ruonia.xlsx,дневная,среднее за месяц,1,основной
5,russia_key_rate,russia_cpi_key_rate.xlsx,месячная,без агрегации,1,чувствительность
6,russia_cpi,russia_cpi.xlsx,месячная,без агрегации,1,чувствительность
7,russia_m2,russia_m2.xlsx,месячная,без агрегации,1,чувствительность
8,russia_credit,russia_private_credit.xlsx,месячная,без агрегации,1,чувствительность
9,russia_retail,russia_retail_yoy.xlsx,месячная,без агрегации,1,чувствительность


**Внутренние признаки vs основные внешние vs чувствительность:**

,Набор признаков,Модель,Горизонт,MAE,RMSE,Отн. MAE к naive,Число прогнозов
16,Основные внешние,random_forest,1,6.543,10.591,1.442,28
17,Основные внешние,hist_gradient_boosting,1,7.915,11.162,1.744,28
18,Основные внешние,elastic_net,1,18.586,19.981,4.096,28
19,Основные внешние,ridge,1,31.465,32.544,6.933,28
20,Основные внешние,random_forest,3,6.571,10.320,1.059,26
21,Основные внешние,hist_gradient_boosting,3,7.988,11.463,1.287,26
22,Основные внешние,elastic_net,3,17.765,19.676,2.863,26
23,Основные внешние,ridge,3,30.932,32.411,4.985,26
24,Основные внешние,random_forest,6,6.927,10.927,1.012,24
25,Основные внешние,hist_gradient_boosting,6,8.837,12.227,1.291,24


## 3) Этап 3: вероятностный прогноз и проверка общих точек прогноза

В проекте используются фактические результаты квантильных моделей на общей сетке точек прогноза. Новые прогнозы на этом этапе в ноутбуке не строятся: используются уже полученные вероятностные результаты.

QAR, QR-X, QGB и эмпирическая базовая модель сравниваются на одной и той же общей сетке точек прогноза.

In [4]:
common = build_stage3_common_origin_summary(series, horizons=(1, 3, 6, 12))
display_ru(common.round(3))

leader = build_stage3_probabilistic_leaderboard(series, horizons=(1, 3, 6, 12))
display_ru(leader.sort_values(['horizon', 'pinball']).round(3))


,Горизонт,Внутренние признаки: n,Внешние признаки: n,Общее n,Первая точка прогноза,Последняя точка прогноза,Число прогнозов,Совпадают точки прогноза
0,1,454,192,192,2010-02-01,2026-02-01,192,Нет
1,3,452,190,190,2010-02-01,2025-12-01,190,Нет
2,6,449,188,188,2010-02-01,2025-09-01,188,Нет
3,12,443,182,182,2010-02-01,2025-03-01,182,Нет


,Модель,Набор признаков,Горизонт,pinball,CRPS,Покрытие 90%,Ширина 90%,Winkler,MAE медианы,RMSE медианы,Число прогнозов,Доля пересечений до коррекции
2,qar,внутренние,1,5.476,3.596,0.893,16.664,36.804,4.282,7.187,28,0.214
6,qr_x,внутренние,1,5.476,3.596,0.893,16.664,36.804,4.282,7.187,28,0.214
0,baseline,основные внешние,1,5.863,3.796,0.964,18.366,35.257,4.667,7.319,28,0.000
1,baseline,внутренние,1,5.863,3.796,0.964,18.366,35.257,4.667,7.319,28,0.000
4,qgb,внутренние,1,6.631,4.003,0.893,25.508,34.449,6.422,9.093,28,0.929
3,qgb,основные внешние,1,8.963,5.576,0.929,28.756,44.766,6.082,9.103,28,0.964
5,qr_x,основные внешние,1,9.204,5.896,0.964,31.517,50.065,7.176,9.816,28,0.893
11,qgb,внутренние,3,9.220,6.424,0.929,28.204,37.514,9.515,12.258,28,0.964
7,baseline,основные внешние,3,11.327,6.918,0.964,37.988,55.427,6.784,10.156,28,0.000
8,baseline,внутренние,3,11.327,6.918,0.964,37.988,55.427,6.784,10.156,28,0.000


## 4) Этап 4: нейросетевые модели и ограничение фиксированной сетки прогнозов

На финальном этапе 4 используется фиксированная годовая сетка оценки. В сохранённых метаданных указано `stride_months = 12`, поэтому отложенная тестовая выборка мала: на каждый горизонт доступно только пять точек прогноза.

Результаты нейросетей носят исследовательский характер, поскольку при фиксированной годовой сетке оценки доступно только пять отложенных точек прогноза на каждый горизонт.

Это не плотный ежемесячный бэктест.

In [5]:
pred = pd.read_csv(B / 'outputs' / 'stage4_eval' / 'predictions.csv')
test = pred[pred['period'] == 'test'].copy()
origins = sorted(test['origin'].drop_duplicates().tolist())
stage4_frame = pd.DataFrame([{
    'Шаг, мес.': 12,
    'Число тестовых точек': len(origins),
    'Прогнозов на горизонт': len(origins),
    'Первая точка прогноза': origins[0],
    'Последняя точка прогноза': origins[-1],
    'Примечание': 'Результаты нейросетей носят исследовательский характер: при фиксированной годовой сетке доступно только пять отложенных точек прогноза на каждый горизонт.'
}])
display(stage4_frame)

origins_by_model = (
    test[test['model'].isin(['naive', 'mlp', 'rnn', 'lstm', 'gru', 'cnn', 'qrnn'])]
    .groupby('model')['origin'].nunique()
    .reset_index(name='n_origins')
    .sort_values('model')
)
display_ru(origins_by_model)


,"Шаг, мес.",Число тестовых точек,Прогнозов на горизонт,Первая точка прогноза,Последняя точка прогноза,Примечание
0,12,5,5,2020-04-01,2024-04-01,Результаты нейросетей носят исследовательский характер...


,Модель,Число точек прогноза
0,cnn,5
1,gru,5
2,lstm,5
3,mlp,5
4,naive,5
5,qrnn,5
6,rnn,5


## 5) Сравнение семейств моделей

Ниже приведено компактное сравнение семейств, а не абсолютный рейтинг. Семейства оцениваются на разных вневыборочных периодах, поэтому напрямую сопоставлять их метрики нельзя.

In [6]:
cross_family = pd.DataFrame([
    {
        'family': 'Этап 1',
        'best_model': 'классический ориентир из бэктеста на общей сетке прогнозов',
        'evaluation_sample': 'статистический бэктест на общей сетке прогнозов',
        'horizon': '1, 3, 6, 12',
        'metric': 'MAE / RMSE / MAPE',
        'value': 'см. таблицу этапа 1 выше',
        'n_forecasts': 'только общая сетка прогнозов',
        'comparability': 'НАПРЯМУЮ НЕСОПОСТАВИМО'
    },
    {
        'family': 'Этап 2',
        'best_model': 'лучшая прямая ML-/макро-модель этапа 2',
        'evaluation_sample': 'общая сетка прямых прогнозов с признаками',
        'horizon': '1, 3, 6, 12',
        'metric': 'MAE / RMSE',
        'value': 'см. таблицу этапа 2 выше',
        'n_forecasts': 'только общая сетка прогнозов',
        'comparability': 'НАПРЯМУЮ НЕСОПОСТАВИМО'
    },
    {
        'family': 'Этап 3',
        'best_model': 'лучшая QAR / QR-X / QGB модель вероятностного блока',
        'evaluation_sample': 'квантильная оценка на общей сетке прогнозов',
        'horizon': '1, 3, 6, 12',
        'metric': 'pinball / покрытие / Winkler / MAE медианы',
        'value': 'см. таблицу этапа 3 выше',
        'n_forecasts': 'только общая сетка прогнозов',
        'comparability': 'НАПРЯМУЮ НЕСОПОСТАВИМО'
    },
    {
        'family': 'Этап 4',
        'best_model': 'лучший нейросетевой прогноз на фиксированной годовой сетке',
        'evaluation_sample': 'тестовая выборка с фиксированными годовыми точками прогноза',
        'horizon': '1, 3, 6, 12',
        'metric': 'метрики ошибки прогноза',
        'value': 'см. метаданные / результаты этапа 4',
        'n_forecasts': 'только пять годовых точек прогноза',
        'comparability': 'НАПРЯМУЮ НЕСОПОСТАВИМО'
    }
])

display_ru(
    cross_family[
        ['family', 'best_model', 'evaluation_sample', 'horizon', 'metric', 'value', 'n_forecasts', 'comparability']
    ]
)


,Семейство,Лучшая модель,Выборка оценки,Горизонт,Метрика,Значение,Число прогнозов,Сопоставимость
0,Этап 1,классический ориентир из бэктеста на общей сетке...,статистический бэктест на общей сетке прогнозов,"1, 3, 6, 12",MAE / RMSE / MAPE,см. таблицу этапа 1 выше,только общая сетка прогнозов,НАПРЯМУЮ НЕСОПОСТАВИМО
1,Этап 2,лучшая прямая ML-/макро-модель этапа 2...,общая сетка прямых прогнозов с признаками,"1, 3, 6, 12",MAE / RMSE,см. таблицу этапа 2 выше,только общая сетка прогнозов,НАПРЯМУЮ НЕСОПОСТАВИМО
2,Этап 3,лучшая QAR / QR-X / QGB модель вероятностного блока...,квантильная оценка на общей сетке прогнозов,"1, 3, 6, 12",pinball / покрытие / Winkler / MAE медианы,см. таблицу этапа 3 выше,только общая сетка прогнозов,НАПРЯМУЮ НЕСОПОСТАВИМО
3,Этап 4,лучший нейросетевой прогноз на фиксированной годовой сетке...,тестовая выборка с фиксированными годовыми точками прогноза,"1, 3, 6, 12",метрики ошибки прогноза,см. метаданные / результаты этапа 4,только пять годовых точек прогноза,НАПРЯМУЮ НЕСОПОСТАВИМО


## 3) Этап 3: вероятностный блок и общая сетка точек прогноза

Для сравнения вероятностных и точечных моделей используется единая сетка общих точек прогноза.
Это не все возможные даты, а одинаковые точки прогноза внутри каждого горизонта.
Метрики строятся по квантилям `[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]`: pinball loss, CRPS, покрытие и Winkler score.

## 1C) ACF / PACF

Для анализа стационарной части ряда используются лог-доходности — наиболее простое устойчивое преобразование в диагностике этапа 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ret = np.log(series).diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(ret, lags=24, ax=axes[0])
axes[0].set_title('ACF: лог-доходность Brent')
plot_pacf(ret, lags=24, ax=axes[1])
axes[1].set_title('PACF: лог-доходность Brent')
plt.tight_layout()
plt.show()


In [8]:
common = build_stage3_common_origin_summary(series, horizons=(1, 3, 6, 12))
display_ru(common.round(3))

leader = build_stage3_probabilistic_leaderboard(series, horizons=(1, 3, 6, 12))
best = (
    leader.sort_values(['horizon', 'pinball'])
    .groupby('horizon', as_index=False)
    .head(3)
    [['model', 'feature_set', 'horizon', 'pinball', 'CRPS', 'coverage_90', 'width_90', 'median_MAE']]
)
display_ru(best.round(3))


,Горизонт,Внутренние признаки: n,Внешние признаки: n,Общее n,Первая точка прогноза,Последняя точка прогноза,Число прогнозов,Совпадают точки прогноза
0,1,454,192,192,2010-02-01,2026-02-01,192,Нет
1,3,452,190,190,2010-02-01,2025-12-01,190,Нет
2,6,449,188,188,2010-02-01,2025-09-01,188,Нет
3,12,443,182,182,2010-02-01,2025-03-01,182,Нет


,Модель,Набор признаков,Горизонт,pinball,CRPS,Покрытие 90%,Ширина 90%,MAE медианы
2,qar,внутренние,1,5.476,3.596,0.893,16.664,4.282
6,qr_x,внутренние,1,5.476,3.596,0.893,16.664,4.282
0,baseline,основные внешние,1,5.863,3.796,0.964,18.366,4.667
11,qgb,внутренние,3,9.220,6.424,0.929,28.204,9.515
7,baseline,основные внешние,3,11.327,6.918,0.964,37.988,6.784
8,baseline,внутренние,3,11.327,6.918,0.964,37.988,6.784
17,qgb,основные внешние,6,13.960,8.303,0.964,53.112,13.010
18,qgb,внутренние,6,14.953,10.247,0.964,48.974,17.672
14,baseline,основные внешние,6,15.267,8.759,0.964,54.192,6.337
24,qgb,основные внешние,12,14.603,11.524,0.481,34.176,22.889


## 4) Этап 4: нейросети и явное ограничение по размеру выборки

Финальный этап 4 использует фиксированную сетку точек прогноза с шагом 12 месяцев:
это не плотный ежемесячный бэктест. По метаданным `stage4_eval/metadata.json`
используются `stride_months = 12` и `epochs_max = 20` в быстром режиме, а тестовый блок
содержит лишь небольшое число независимых точек прогноза. Поэтому результаты этапа 4
следует трактовать как ограниченную ретроспективную проверку, а не как полноценную
ежемесячную сравнительную панель.

In [9]:
pred = pd.read_csv(B / 'outputs' / 'stage4_eval' / 'predictions.csv')
test = pred[pred['period'] == 'test'].copy()
origins = sorted(test['origin'].drop_duplicates().tolist())
stage4_overview = pd.DataFrame([{
    'Режим': {'fast': 'быстрый'}.get(stage4_meta['mode'], stage4_meta['mode']),
    'Шаг, мес.': stage4_meta['stride_months'],
    'Максимум эпох': stage4_meta['epochs_max'],
    'Число тестовых точек': len(origins),
    'Первая точка прогноза': origins[0],
    'Последняя точка прогноза': origins[-1],
    'Моделей в тесте': test['model'].nunique(),
}])
display(stage4_overview)

origins_by_model = (
    test[test['model'].isin(['naive', 'mlp', 'rnn', 'lstm', 'gru', 'cnn', 'qrnn'])]
    .groupby('model')['origin'].nunique()
    .reset_index(name='n_origins')
    .sort_values('model')
)
display_ru(origins_by_model)


,Режим,"Шаг, мес.",Максимум эпох,Число тестовых точек,Первая точка прогноза,Последняя точка прогноза,Моделей в тесте
0,быстрый,12,20,5,2020-04-01,2024-04-01,33


,Модель,Число точек прогноза
0,cnn,5
1,gru,5
2,lstm,5
3,mlp,5
4,naive,5
5,qrnn,5
6,rnn,5


## 2) Этап 1: эконометрика и классические базовые модели

В этом разделе используются уже рассчитанные результаты проекта для классического семейства моделей. Вывод оставлен компактным и содержит только доступные результаты этапа 1.

In [10]:
from task1_brent.brent_lib.backtest import evaluate, metrics_table
from task1_brent.brent_lib.models import STATISTICAL_MODELS, DIRECT_MODELS

errors = evaluate(
    series,
    horizons=(1, 3, 6, 12),
    initial_window=120,
    direct_start=series.index[120],
    stat=True,
)
leader = metrics_table(errors, common_origins=True)
leader = leader[leader['horizon'].isin([1, 3, 6, 12])].sort_values(['horizon', 'MAE']).reset_index(drop=True)

display(pd.DataFrame({'Доступные классические модели': sorted({m.name for m in STATISTICAL_MODELS})}))
display(pd.DataFrame({'Доступные прямые модели': sorted({m.name for m in DIRECT_MODELS})}))
display_ru(leader[['horizon', 'model', 'MAE', 'RMSE', 'MAPE_%', 'rel_MAE_vs_naive']].round(3))


,Доступные классические модели
0,ar_1
1,drift
2,ets_log
3,historical_mean
4,holt
5,mean_reversion
6,naive
7,selected_ar
8,selected_arima
9,selected_arma


,Доступные прямые модели
0,hgb_direct
1,ridge_direct


,Горизонт,Модель,MAE,RMSE,"MAPE, %",Отн. MAE к naive
0,1,drift,4.203,5.968,7.532,0.997
1,1,naive,4.216,5.962,7.545,1.000
2,1,selected_arma,4.227,5.810,7.665,1.003
3,1,holt,4.239,6.000,7.668,1.006
4,1,selected_arima,4.245,5.823,7.689,1.007
5,1,ses,4.250,6.126,7.700,1.008
6,1,ar_1,4.256,6.014,7.667,1.010
7,1,ets_log,4.261,6.008,7.660,1.011
8,1,selected_ar,4.269,5.872,7.636,1.013
9,1,mean_reversion,4.522,6.297,7.883,1.072


## 6) Ограничения

- Месячная выборка невелика для долгосрочного экономического прогнозирования.
- Нельзя исключить структурные сдвиги, особенно с учётом ограниченного числа наблюдений после 2022 года.
- Предположения о лагах публикации макроданных заданы явно, но не проверялись в режиме реального времени.
- Для части макропоказателей возможна обратная причинность относительно цены нефти.
- Разные вневыборочные периоды между семействами моделей нельзя напрямую сопоставлять.
- На этапе 4 для нейросетей доступно только пять отложенных точек прогноза при фиксированной годовой сетке.
- Качество прогноза не является доказательством причинной связи.
- Небольшие различия метрик сами по себе не означают статистически значимого превосходства.

## 5) Выводы

- Цена Brent нестационарна в уровнях, тогда как первые разности и лог-доходности стационарны.
- Простые базовые модели остаются конкурентоспособными на коротких горизонтах: drift и naive близки к лучшим значениям MAE.
- Mean reversion не является универсальным победителем: в отдельных спецификациях модель конкурентоспособна, но не доминирует на всех горизонтах.
- Этап 2 показывает пользу дополнительных признаков только при честном сравнении на общей вневыборочной сетке, особенно на средних горизонтах.
- Этап 3 подтверждает, что вероятностные модели нужно сравнивать на одной и той же сетке точек прогноза.
- Этап 4 полезен как дополнительная проверка, но выводы по нейросетям ограничены пятью тестовыми точками на горизонт.
- Семейства моделей нельзя ранжировать так, будто они оценивались на одинаковых вневыборочных периодах.
- Результаты проекта лучше интерпретировать как несколько честных блоков проверки, а не как поиск единственного победителя.
- Небольшая разница MAE не является достаточным доказательством статистически значимого преимущества.
- Главный итог — прозрачное сравнение прогнозов с явным учётом ограничений выборки и без завышенных выводов.